# 04 - Train Models

This notebook trains and compares several classifiers for the differential diagnosis task. The output is designed to be presentation-ready and compatible with the Healix ML predictor path.

In [10]:
import sys
from pathlib import Path

_nb = Path.cwd().resolve()
for _c in [_nb, _nb.parent, _nb.parent.parent]:
    if (_c / "app" / "main.py").exists():
        sys.path.insert(0, str(_c))
        break

from training.notebooks.bootstrap import require_ddxplus

ctx = require_ddxplus()
repo_root = ctx.repo_root
raw_path = ctx.raw_train_csv
processed_dir = ctx.processed_dir
training_models_dir = ctx.training_models_dir
runtime_models_dir = ctx.runtime_models_dir

import json
import time

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

X_train = pd.read_csv(processed_dir / "X_train.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")["target"].astype(np.int16)
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_test = pd.read_csv(processed_dir / "y_test.csv")["target"].astype(np.int16)

evidence_cols = [c for c in X_train.columns if c not in ("AGE", "SEX_M")]
for frame in (X_train, X_test):
    frame[evidence_cols] = frame[evidence_cols].astype(np.int8)
    frame["SEX_M"] = frame["SEX_M"].astype(np.int8)
    frame["AGE"] = frame["AGE"].astype(np.float32)

TRAIN_SAMPLE_WEIGHT = compute_sample_weight("balanced", y_train)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Balanced sample weights enabled for boosting models")


Training set shape: (15000, 225)
Test set shape: (5000, 225)
Balanced sample weights enabled for boosting models


In [11]:
# Helper evaluation function

def evaluate_model(model, X_train, y_train, X_test, y_test, sample_weight=None):
    start = time.perf_counter()
    if sample_weight is None:
        model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train, sample_weight=sample_weight)
    elapsed = time.perf_counter() - start
    preds = model.predict(X_test)
    return {
        "model": model,
        "train_time_sec": round(elapsed, 3),
        "accuracy": round(accuracy_score(y_test, preds), 4),
        "precision_macro": round(precision_score(y_test, preds, average="macro", zero_division=0), 4),
        "recall_macro": round(recall_score(y_test, preds, average="macro", zero_division=0), 4),
        "f1_macro": round(f1_score(y_test, preds, average="macro", zero_division=0), 4),
    }

In [12]:
# 1) Logistic Regression baseline (class_weight='balanced' per CLAUDE.md)
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, random_state=42, class_weight="balanced"),
)
logreg_result = evaluate_model(logreg, X_train, y_train, X_test, y_test)
print("Logistic Regression:", logreg_result)

Logistic Regression: {'model': Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=42))]), 'train_time_sec': 3.646, 'accuracy': 0.9942, 'precision_macro': 0.9946, 'recall_macro': 0.9924, 'f1_macro': 0.9927}


In [13]:
# 2) Random Forest
rf = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced",
)
rf_result = evaluate_model(rf, X_train, y_train, X_test, y_test)
print("Random Forest:", rf_result)

Random Forest: {'model': RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42), 'train_time_sec': 1.201, 'accuracy': 0.9936, 'precision_macro': 0.9937, 'recall_macro': 0.991, 'f1_macro': 0.9917}


In [14]:
# 3) XGBoost with predict_proba support
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
xgb_result = evaluate_model(
    xgb, X_train, y_train, X_test, y_test, sample_weight=TRAIN_SAMPLE_WEIGHT,
)
print("XGBoost:", xgb_result)

XGBoost: {'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=-1, num_parallel_tree=None, objective='multi:softprob', ...), 'train_time_sec': 27.495, 'accuracy': 0.9924, 'precision_macro': 0.9912, 'recall_macro': 0.9899, 'f1_macro': 0.9902}


In [15]:
# 4) LightGBM (optional, only if installed)
try:
    from lightgbm import LGBMClassifier
    lgbm = LGBMClassifier(n_estimators=200, random_state=42, verbosity=-1)
    lgbm_result = evaluate_model(
        lgbm, X_train, y_train, X_test, y_test, sample_weight=TRAIN_SAMPLE_WEIGHT,
    )
    print("LightGBM:", lgbm_result)
except Exception as exc:
    print("LightGBM unavailable in this environment:", exc)
    lgbm_result = None

LightGBM: {'model': LGBMClassifier(n_estimators=200, random_state=42, verbosity=-1), 'train_time_sec': 6.049, 'accuracy': 0.9932, 'precision_macro': 0.9728, 'recall_macro': 0.9643, 'f1_macro': 0.9673}


In [16]:
# Hyperparameter tuning for XGBoost (small search)
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
}
search = RandomizedSearchCV(
    XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", random_state=42, n_jobs=8),
    param_distributions=param_grid,
    n_iter=1,
    cv=2,
    scoring="f1_macro",
    random_state=42,
    n_jobs=1,
)
search.fit(X_train, y_train, sample_weight=TRAIN_SAMPLE_WEIGHT)
print("Best XGBoost params:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 4))

Best XGBoost params: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05}
Best CV F1: 0.9748


In [17]:
# Create a differential diagnosis ranking example
best_model = search.best_estimator_ if hasattr(search, "best_estimator_") else xgb
sample_row = X_test.iloc[0].to_frame().T
proba = best_model.predict_proba(sample_row)[0]
classes = np.array(sorted(set(y_train)))
# Use the trained label encoder from the preprocessing notebook if available
try:
    import joblib
    label_encoder = joblib.load(repo_root / "training" / "models" / "label_encoder.joblib")
    labels = label_encoder.inverse_transform(classes)
except Exception:
    labels = [str(c) for c in classes]

ranking = sorted(zip(labels, proba), key=lambda item: item[1], reverse=True)[:5]
print("Top differential diagnosis predictions:")
for disease, probability in ranking:
    print(f"- {disease}: {probability:.3f}")

Top differential diagnosis predictions:
- Panic attack: 0.994
- Boerhaave: 0.003
- Acute otitis media: 0.000
- Myocarditis: 0.000
- Possible NSTEMI / STEMI: 0.000


## Training summary

The notebook compares multiple classifiers and produces probabilities needed for differential diagnosis ranking. The XGBoost model is retained because it supports `predict_proba()` and is suitable for ranking possible conditions.